# 1972. First and Last Call On the Same Day
**Level:** Hard

## Problem Description
We need to report the IDs of users who had the **first and last call with the same person on any day**.  

- Each row in the `Calls` table contains information about a call between `caller_id` and `recipient_id` at a specific `call_time`.  
- If a user’s first and last call on a given day is with the same person, that user should be included in the result.  
- Return the result table in any order.

---

## Schema

### Table: Calls
| Column Name  | Type     | Description                                      |
|--------------|----------|--------------------------------------------------|
| caller_id    | INT      | ID of the caller                                 |
| recipient_id | INT      | ID of the recipient                              |
| call_time    | DATETIME | Timestamp of the call                            |

**Primary Key:** `(caller_id, recipient_id, call_time)`

---

## Sample Data

### Calls
| caller_id | recipient_id | call_time           |
|-----------|--------------|---------------------|
| 8         | 4            | 2021-08-24 17:46:07 |
| 4         | 8            | 2021-08-24 19:57:13 |
| 5         | 1            | 2021-08-11 05:28:44 |
| 8         | 3            | 2021-08-17 04:04:15 |
| 11        | 3            | 2021-08-17 13:07:00 |
| 8         | 11           | 2021-08-17 22:22:22 |

---

## Expected Output
| user_id |
|---------|
| 1       |
| 4       |
| 5       |
| 8       |

### Explanation
- On **2021-08-24**, user 8’s first and last call was with user 4 → include user 8.  
- On **2021-08-24**, user 4’s first and last call was with user 8 → include user 4.  
- On **2021-08-11**, user 1 and user 5 had only one call with each other → both included.  
- On **2021-08-17**, user 8’s first call was with user 3 and last call was with user 11 → not included.  

---

## PySpark Code: Create DataFrame and Temp View

```python



In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType
from datetime import datetime

# Schema for Calls
calls_schema = StructType([
    StructField("caller_id", IntegerType(), False),
    StructField("recipient_id", IntegerType(), False),
    StructField("call_time", TimestampType(), False)
])

# Data for Calls
calls_data = [
    (8, 4, datetime(2021, 8, 24, 17, 46, 7)),
    (4, 8, datetime(2021, 8, 24, 19, 57, 13)),
    (5, 1, datetime(2021, 8, 11, 5, 28, 44)),
    (8, 3, datetime(2021, 8, 17, 4, 4, 15)),
    (11, 3, datetime(2021, 8, 17, 13, 7, 0)),
    (8, 11, datetime(2021, 8, 17, 22, 22, 22))
]

# Create DataFrame
calls_df = spark.createDataFrame(calls_data, calls_schema)

# Register Temp View
calls_df.createOrReplaceTempView("Calls")

# Quick check
calls_df.show()

In [0]:
%sql

WITH cte AS (
		SELECT caller_id AS user_id_1,
			recipient_id AS user_id_2,
			call_time
		FROM Calls
		
		UNION
		
		SELECT recipient_id AS user_id_1,
			caller_id AS user_id_2,
			call_time
		FROM Calls
		),
	cte2(SELECT DISTINCT user_id_2, first_value(user_id_1) OVER (
			PARTITION BY user_id_2,
			DATE (call_time) ORDER BY call_time ASC
			) AS f, first_value(user_id_1) OVER (
			PARTITION BY user_id_2,
			DATE (call_time) ORDER BY call_time DESC
			) AS l, DATE (call_time) AS call_date FROM cte ORDER BY call_date ASC)

SELECT DISTINCT user_id_2 AS user_id
FROM cte2
WHERE f = l


# Documentation: First and Last Call on the Same Day Query

## Step 1: Create a Symmetric Call Representation (CTE)
```markdown


```sql
WITH cte AS (
    SELECT caller_id AS user_id_1,
           recipient_id AS user_id_2,
           call_time
    FROM Calls
    
    UNION
    
    SELECT recipient_id AS user_id_1,
           caller_id AS user_id_2,
           call_time
    FROM Calls
)
```
- The `Calls` table records calls in one direction (`caller_id → recipient_id`).  
- To analyze calls from both perspectives, we **duplicate the data** by swapping caller and recipient.  
- The `UNION` ensures we capture both directions, creating a symmetric view of calls.  
- Result: Each call is represented twice, once for each participant.

---

## Step 2: Identify First and Last Call Partners per Day (CTE2)
```sql
cte2 AS (
    SELECT DISTINCT 
           user_id_2,
           FIRST_VALUE(user_id_1) OVER (
               PARTITION BY user_id_2, DATE(call_time) 
               ORDER BY call_time ASC
           ) AS f,
           FIRST_VALUE(user_id_1) OVER (
               PARTITION BY user_id_2, DATE(call_time) 
               ORDER BY call_time DESC
           ) AS l,
           DATE(call_time) AS call_date
    FROM cte
    ORDER BY call_date ASC
)
```
- For each `user_id_2` (the focal user) and each day (`DATE(call_time)`):
  - `FIRST_VALUE(... ORDER BY call_time ASC)` → the **first person** they called that day.  
  - `FIRST_VALUE(... ORDER BY call_time DESC)` → the **last person** they called that day.  
- `DISTINCT` ensures we don’t duplicate rows unnecessarily.  
- This step produces, for each user and day:
  - `f` → first call partner.  
  - `l` → last call partner.  
  - `call_date` → the date of the calls.

---

## Step 3: Filter Users Whose First and Last Call Partner Match
```sql
SELECT DISTINCT user_id_2 AS user_id
FROM cte2
WHERE f = l
```
- We select users where the **first call partner (`f`) equals the last call partner (`l`)** on a given day.  
- This means their first and last call of the day was with the same person.  
- `DISTINCT` ensures each user appears only once in the final result.

---

## Final Output
- The query returns a list of `user_id`s who had their **first and last call with the same person on any given day**.  
- The result is unordered (can be returned in any order).

---

## Key Insights
1. **Symmetric representation** of calls is necessary because calls involve two users.  
2. **Window functions (`FIRST_VALUE`)** allow us to efficiently capture the first and last call partner per day.  
3. **Filtering on equality (`f = l`)** isolates the users who meet the condition.  
4. The query is concise compared to approaches that might require multiple joins or subqueries.
```